# Data Load

In [1]:
import numpy as np
from pathlib import Path


def load_dataset(path="./processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


X, y, patient = load_dataset()

# Model

In [2]:
"""
PyTorch implementation of

    Zhang et al., "Adversarial Representation Learning for Robust
    Patient-Independent Epileptic Seizure Detection",
    IEEE J. Biomedical and Health Informatics, 24(10), 2020.

Generalized to arbitrary window sizes. The paper uses (time=250, chan=22);
this version is configured for your data: (chan=21, time=768) = (21, 256*3).

--------------------------------------------------------------------------
INPUT CONVENTION
    Feed windows exactly as they come out of your loader:
        e : [B, n_chan, n_time]      e.g. [B, 21, 768]
    Internally this is transposed to [B, 1, n_time, n_chan] so that time is
    the conv "height" and channels the "width" (matching the paper, where the
    per-channel attention has one weight per channel).

WHY LAZY LAYERS
    The flatten dimension after the 4 conv/pool stages depends on the window
    size (1792 for the paper's 250x22, 6144 for your 768x21). Using
    nn.LazyLinear lets the first FC and the attention layer infer their input
    size automatically, so the model adapts to any window shape with no manual
    arithmetic. All lazy params are materialized in __init__ (a dummy forward),
    so you can build the optimizer immediately after construction.
--------------------------------------------------------------------------
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------
# 1. EEG decomposition (encoder + decoder), Eq. 2-5
# ----------------------------------------------------------------------
class Decomposer(nn.Module):
    """Encode E -> latent (S̄ or P̄), decode latent -> full component (S or P).

    Encoder : conv (SAME, stride 1) -> ReLU -> [2,1] max-pool over time
              [B,1,T,C] -> [B,4,T,C] -> [B,4,T//2,C]        (= S̄ / P̄)
    Decoder : transposed conv restores time T//2 -> T, 4 -> 1 filter
              [B,4,T//2,C] -> [B,1,T,C]                     (= S / P)
    The reconstruction is cropped/padded to match E exactly (handles odd T).
    """

    def __init__(self, n_filters: int = 4):
        super().__init__()
        self.enc = nn.Conv2d(1, n_filters, kernel_size=3, stride=1, padding="same")
        self.pool = nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))
        self.dec = nn.ConvTranspose2d(n_filters, 1, kernel_size=(2, 1), stride=(2, 1))

    def forward(self, e):                     # e: [B,1,T,C]
        latent = F.relu(self.enc(e))          # [B,4,T,C]
        latent = self.pool(latent)            # [B,4,T//2,C]   S̄ / P̄
        recon = F.relu(self.dec(latent))      # [B,1,~T,C]     S / P
        recon = _match_time(recon, e.shape[2])  # force time dim == T
        return latent, recon


def _match_time(x, t_target):
    """Crop or zero-pad x along the time axis (dim 2) to length t_target."""
    t = x.shape[2]
    if t == t_target:
        return x
    if t > t_target:
        return x[:, :, :t_target, :]
    return F.pad(x, (0, 0, 0, t_target - t))   # pad time at the end


# ----------------------------------------------------------------------
# 2. Shared conv backbone: 4 conv+pool stages -> FC(300) -> FC(n_chan)
#    Used by both the seizure and patient branches (Fig. 1, Section IV-B).
# ----------------------------------------------------------------------
class ConvBackbone(nn.Module):
    """S̄/P̄ [B,4,T//2,C] -> feature vector of length `feat_dim` (= n_chan)."""

    def __init__(self, feat_dim: int, in_ch: int = 4, dropout: float = 0.2):
        super().__init__()
        # filters 16,32,64,128 ; kernels 3,3,2,2 ; all stride 1, SAME
        self.conv1 = nn.Conv2d(in_ch, 16, kernel_size=3, stride=1, padding="same")
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding="same")
        self.conv3 = nn.Conv2d(32, 64, kernel_size=2, stride=1, padding="same")
        self.conv4 = nn.Conv2d(64, 128, kernel_size=2, stride=1, padding="same")

        self.pool_22 = nn.MaxPool2d(kernel_size=2, stride=2)             # stages 1-3
        self.pool_21 = nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))   # stage 4

        self.flatten = nn.Flatten()
        self.drop = nn.Dropout(dropout)          # paper: 0.8 keep -> 0.2 drop
        self.fc1 = nn.LazyLinear(300)            # in-features inferred at runtime
        self.fc2 = nn.Linear(300, feat_dim)      # -> F²  (length = n_chan)

    def forward(self, x):                        # x: [B,4,T//2,C]
        x = self.pool_22(F.relu(self.conv1(x)))
        x = self.pool_22(F.relu(self.conv2(x)))
        x = self.pool_22(F.relu(self.conv3(x)))
        x = self.pool_21(F.relu(self.conv4(x)))
        x = self.drop(self.flatten(x))
        # Eq. 10 specifies sigmoid FC activations. Kept faithful; switch to
        # ReLU here if you see the loss stall from saturation.
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))           # [B, n_chan]   F²
        return x


# ----------------------------------------------------------------------
# 3. Attention-weighted seizure head, Eq. 11-12
# ----------------------------------------------------------------------
class SeizureHead(nn.Module):
    """att = sigmoid(Linear(E)) gives one weight per channel; ŷ_s = F² · att."""

    def __init__(self, n_chan: int):
        super().__init__()
        self.att = nn.LazyLinear(n_chan)                # E -> n_chan weights

    def forward(self, f2, e_flat):                      # f2:[B,C]  e_flat:[B,T*C]
        att = torch.sigmoid(self.att(e_flat))           # [B,C]  attention (Eq.12)
        logit = (f2 * att).sum(dim=1, keepdim=True)     # [B,1]  ŷ_s (Eq.11)
        return logit, att


# ----------------------------------------------------------------------
# 4. Full model
# ----------------------------------------------------------------------
class AdversarialSeizureNet(nn.Module):
    def __init__(self, n_patients: int, n_chan: int = 21, n_time: int = 768,
                 dropout: float = 0.2, w1: float = 0.5):
        super().__init__()
        self.n_chan, self.n_time = n_chan, n_time
        self.w1, self.w2 = w1, 1.0 - w1          # Eq. 6 reconstruction weights

        self.seizure_decomp = Decomposer()       # E -> S̄, S
        self.patient_decomp = Decomposer()       # E -> P̄, P

        self.seizure_backbone = ConvBackbone(feat_dim=n_chan, dropout=dropout)
        self.patient_backbone = ConvBackbone(feat_dim=n_chan, dropout=dropout)

        self.seizure_head = SeizureHead(n_chan)
        self.patient_head = nn.Linear(n_chan, n_patients)   # -> C classes

        self._materialize()                      # init all LazyLinear params

    def _materialize(self):
        """Run one dummy forward so lazy layers create their weights,
        letting you build the optimizer right after construction."""
        was_training = self.training
        self.eval()
        with torch.no_grad():
            self(torch.zeros(1, self.n_chan, self.n_time))
        self.train(was_training)

    def forward(self, e):                        # e: [B, n_chan, n_time]
        if e.dim() == 3:
            e = e.transpose(1, 2).unsqueeze(1)   # -> [B,1,n_time,n_chan]
        e_flat = e.flatten(1)                    # [B, n_time*n_chan]

        s_lat, s_rec = self.seizure_decomp(e)    # S̄, S
        p_lat, p_rec = self.patient_decomp(e)    # P̄, P
        e_recon = self.w1 * s_rec + self.w2 * p_rec           # Ê  (Eq. 6)

        f2_s = self.seizure_backbone(s_lat)
        seizure_logit, att = self.seizure_head(f2_s, e_flat)  # [B,1], [B,C]

        f2_p = self.patient_backbone(p_lat)
        patient_logits = self.patient_head(f2_p)              # [B,C_patients]

        return {
            "seizure_logit": seizure_logit,      # [B,1]  BCE-with-logits target
            "patient_logits": patient_logits,    # [B,C]  CE target
            "e_recon": e_recon,                  # [B,1,n_time,n_chan]
            "e": e,                              # [B,1,n_time,n_chan]
            "attention": att,                    # [B,n_chan]
            "S": s_rec, "P": p_rec,
            "S_latent": s_lat, "P_latent": p_lat,
        }


# ----------------------------------------------------------------------
# 5. Losses and the two-pass training step (Section III-E, Algorithm 1)
# ----------------------------------------------------------------------
def compute_losses(out, y_seizure, y_patient):
    l_d = F.mse_loss(out["e_recon"], out["e"])                       # L_D  (Eq. 7)
    l_s = F.binary_cross_entropy_with_logits(                       # L_s  (Eq. 13)
        out["seizure_logit"], y_seizure.float().view(-1, 1))
    l_p = F.cross_entropy(out["patient_logits"], y_patient.long())  # L_p  (Eq. 14)
    return l_d, l_s, l_p


def train_step(model, optimizer, e, y_seizure, y_patient):
    """Pass 1 minimizes L = L_D + L_s + L_p; pass 2 minimizes L_s again to
    raise the seizure task's priority (as the paper does)."""
    model.train()

    optimizer.zero_grad()
    out = model(e)
    l_d, l_s, l_p = compute_losses(out, y_seizure, y_patient)
    (l_d + l_s + l_p).backward()
    optimizer.step()

    optimizer.zero_grad()
    out = model(e)
    _, l_s2, _ = compute_losses(out, y_seizure, y_patient)
    l_s2.backward()
    optimizer.step()

    return {"L_D": l_d.item(), "L_s": l_s.item(),
            "L_p": l_p.item(), "L_s_repeat": l_s2.item()}


def make_optimizer(model, lr: float = 1e-4, weight_decay: float = 1e-4):
    # lr = 1e-4 (Adam); L2 coefficient 0.0001 -> weight_decay (Section III-E)
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


# ----------------------------------------------------------------------
# Smoke test on YOUR window shape: (21, 768)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    torch.manual_seed(0)
    B, C = 8, 13                       # batch, #training patients
    N_CHAN, N_TIME = 21, 256 * 3       # your window shape

    model = AdversarialSeizureNet(n_patients=C, n_chan=N_CHAN, n_time=N_TIME)
    opt = make_optimizer(model)

    e = torch.randn(B, N_CHAN, N_TIME)          # windows as (21, 768)
    y_seiz = torch.randint(0, 2, (B,))
    y_pat = torch.randint(0, C, (B,))

    out = model(e)
    print("seizure_logit :", tuple(out["seizure_logit"].shape))   # (8, 1)
    print("patient_logits:", tuple(out["patient_logits"].shape))  # (8, 13)
    print("e_recon       :", tuple(out["e_recon"].shape))         # (8,1,768,21)
    print("attention     :", tuple(out["attention"].shape))       # (8, 21)

    logs = train_step(model, opt, e, y_seiz, y_pat)
    print("losses        :", {k: round(v, 4) for k, v in logs.items()})
    print("params        :", f"{sum(p.numel() for p in model.parameters()):,}")

/home/gmarihuan/.conda/envs/torchy/lib/python3.10/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


seizure_logit : (8, 1)
patient_logits: (8, 13)
e_recon       : (8, 1, 768, 21)
attention     : (8, 21)
losses        : {'L_D': 1.0874, 'L_s': 3.0711, 'L_p': 2.6355, 'L_s_repeat': 2.2241}
params        : 4,131,503


# Training

In [3]:
import numpy as np


def exclude_patients(X, y, patient, exclude):
    """Drop windows belonging to the given patient id(s).

    Parameters
    ----------
    X, y, patient : arrays from load_dataset()  (aligned along axis 0)
    exclude       : a single patient id or an iterable of ids to remove

    Returns
    -------
    X, y, patient : same format, with the excluded patients removed
    """
    exclude = set(np.atleast_1d(exclude).tolist())
    mask = ~np.isin(patient, list(exclude))
    return X[mask], y[mask], patient[mask]


In [ ]:
import numpy as np
import torch

torch.manual_seed(0)
N_CHAN, N_TIME = 21, 256 * 3
EPOCHS, B = 30, 64
dev = "cuda" if torch.cuda.is_available() else "cpu"


def evaluate(model, Xte, yte):
    model.eval()
    with torch.no_grad():
        logit = model(Xte)["seizure_logit"].squeeze(1)
        prob = torch.sigmoid(logit)
        pred = (prob > 0.5).float()
    tp = ((pred == 1) & (yte == 1)).sum().item()
    tn = ((pred == 0) & (yte == 0)).sum().item()
    fp = ((pred == 1) & (yte == 0)).sum().item()
    fn = ((pred == 0) & (yte == 1)).sum().item()
    acc = (tp + tn) / len(yte)
    sens = tp / (tp + fn) if tp + fn else float("nan")   # recall on seizures
    spec = tn / (tn + fp) if tn + fp else float("nan")
    return acc, sens, spec


results = []
for test_pat in np.unique(patient):
    te = patient == test_pat
    tr = ~te

    # remap training patient ids -> contiguous 0..C-1 (held-out id excluded)
    ptr = patient[tr]
    uniq = np.unique(ptr)
    remap = {p: i for i, p in enumerate(uniq)}
    ptr_idx = np.array([remap[p] for p in ptr])
    C = len(uniq)

    Xtr = torch.tensor(X[tr], dtype=torch.float32)
    ytr = torch.tensor(y[tr], dtype=torch.float32)
    ptr_idx = torch.tensor(ptr_idx, dtype=torch.long)
    Xte = torch.tensor(X[te], dtype=torch.float32, device=dev)
    yte = torch.tensor(y[te], dtype=torch.float32, device=dev)

    model = AdversarialSeizureNet(n_patients=C, n_chan=N_CHAN, n_time=N_TIME).to(dev)
    opt = make_optimizer(model)

    n = len(Xtr)
    for ep in range(EPOCHS):
        perm = torch.randperm(n)
        for i in range(0, n, B):
            idx = perm[i:i + B]
            train_step(model, opt,
                       Xtr[idx].to(dev), ytr[idx].to(dev), ptr_idx[idx].to(dev))

    acc, sens, spec = evaluate(model, Xte, yte)
    results.append((int(test_pat), acc, sens, spec, len(yte)))
    print(f"patient {int(test_pat):2d}: acc={acc:.3f} sens={sens:.3f} "
          f"spec={spec:.3f}  (n={len(yte)})")

# --- summary ---
accs = [r[1] for r in results]
sens = [r[2] for r in results]
spec = [r[3] for r in results]
print("-" * 50)
print(f"MEAN over {len(results)} subjects: "
      f"acc={np.nanmean(accs):.3f}  "
      f"sens={np.nanmean(sens):.3f}  "
      f"spec={np.nanmean(spec):.3f}")

patient  0: acc=0.517 sens=0.986 spec=0.048  (n=292)
patient  1: acc=0.500 sens=1.000 spec=0.000  (n=114)
patient  2: acc=0.500 sens=1.000 spec=0.000  (n=262)
patient  3: acc=0.504 sens=1.000 spec=0.008  (n=250)
patient  4: acc=0.492 sens=0.984 spec=0.000  (n=370)
patient  5: acc=0.500 sens=1.000 spec=0.000  (n=96)
patient  6: acc=0.505 sens=1.000 spec=0.009  (n=214)
patient  7: acc=0.507 sens=0.997 spec=0.016  (n=610)
patient  8: acc=0.511 sens=1.000 spec=0.022  (n=180)
patient  9: acc=0.503 sens=1.000 spec=0.007  (n=292)
patient 10: acc=0.496 sens=0.989 spec=0.004  (n=534)
patient 11: acc=0.497 sens=0.994 spec=0.000  (n=636)
